<a href="https://colab.research.google.com/github/sabarik7180/DE_basics/blob/main/sparksql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder.appName("Data Handling").getOrCreate()

In [6]:
records_data = [
    # Proper records (8 records)
    (1, "Alice Smith", "CA", "2023-01-15", 125.50, True),
    (2, "Bob Johnson", "NY", "2023-02-20", 75.00, True),
    (3, "Charlie Brown", "TX", "2023-03-01", 200.25, True),
    (4, "Diana Prince", "FL", "2023-04-10", 50.00, True),
    (5, "Eve Adams", "WA", "2023-05-05", 300.75, True),
    (6, "Frank White", "IL", "2023-06-12", 99.99, True),
    (7, "Grace Lee", "GA", "2023-07-22", 15.20, True),
    (8, "Henry King", "AZ", "2023-08-01", 180.00, True),
    # Improper records (examples of not proper data)
    (9, "Ivy Queen", "PA", "2023/09/01", 60.50, True),  # Incorrect date format
    (10, "Jack Black", "OH", "10-10-2023", 110.00, True), # Incorrect date format
    (11, "Karen Green", "MI", "2023-11-01", "NaN", True), # Price spent not a number
    (12, "Liam Blue", "CO", "2023-12-05", None, False), # price is none , marked as incorrect
    (13, "Mia Stone", "VA", None, 45.00, False), # Missing date, marked as incorrect
    (14, "Nate River", "NC", "2024-01-01", 88.88, "True"), # is_correct_record not boolean
    (15, "Olivia Gold", "SC", "2024-02-14", 150, False), # Price spent as int, marked incorrect
    (16, "Peter Parker", "CA", "2024-03-20", None, False), # Missing price, marked as incorrect
    (17, "Quinn Red", "OR", "2024-04-01", 70.00, 1),  #boolean contains 1 instead of true
    (18, None, "NV", "2024-05-01", 90.00, True) # Missing name
]

columns = ['id' , 'name', 'state', 'date', 'price_spent', 'is_correct_record']
# You can print the data to verify
# for record in records_data:
#     print(record)

In [7]:
df = spark.createDataFrame(data=records_data, schema=columns)
df.show()

+---+-------------+-----+----------+-----------+-----------------+
| id|         name|state|      date|price_spent|is_correct_record|
+---+-------------+-----+----------+-----------+-----------------+
|  1|  Alice Smith|   CA|2023-01-15|      125.5|             true|
|  2|  Bob Johnson|   NY|2023-02-20|       75.0|             true|
|  3|Charlie Brown|   TX|2023-03-01|     200.25|             true|
|  4| Diana Prince|   FL|2023-04-10|       50.0|             true|
|  5|    Eve Adams|   WA|2023-05-05|     300.75|             true|
|  6|  Frank White|   IL|2023-06-12|      99.99|             true|
|  7|    Grace Lee|   GA|2023-07-22|       15.2|             true|
|  8|   Henry King|   AZ|2023-08-01|      180.0|             true|
|  9|    Ivy Queen|   PA|2023/09/01|       60.5|             true|
| 10|   Jack Black|   OH|10-10-2023|      110.0|             true|
| 11|  Karen Green|   MI|2023-11-01|        NaN|             true|
| 12|    Liam Blue|   CO|2023-12-05|       NULL|            fa

In [8]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- date: string (nullable = true)
 |-- price_spent: string (nullable = true)
 |-- is_correct_record: string (nullable = true)



In [9]:
from pyspark.sql.functions import *

df = df.withColumn('name_upper' , upper(df.name))
df.show()

+---+-------------+-----+----------+-----------+-----------------+-------------+
| id|         name|state|      date|price_spent|is_correct_record|   name_upper|
+---+-------------+-----+----------+-----------+-----------------+-------------+
|  1|  Alice Smith|   CA|2023-01-15|      125.5|             true|  ALICE SMITH|
|  2|  Bob Johnson|   NY|2023-02-20|       75.0|             true|  BOB JOHNSON|
|  3|Charlie Brown|   TX|2023-03-01|     200.25|             true|CHARLIE BROWN|
|  4| Diana Prince|   FL|2023-04-10|       50.0|             true| DIANA PRINCE|
|  5|    Eve Adams|   WA|2023-05-05|     300.75|             true|    EVE ADAMS|
|  6|  Frank White|   IL|2023-06-12|      99.99|             true|  FRANK WHITE|
|  7|    Grace Lee|   GA|2023-07-22|       15.2|             true|    GRACE LEE|
|  8|   Henry King|   AZ|2023-08-01|      180.0|             true|   HENRY KING|
|  9|    Ivy Queen|   PA|2023/09/01|       60.5|             true|    IVY QUEEN|
| 10|   Jack Black|   OH|10-

In [10]:
df.filter(df.state.startswith('C')).show()

+---+------------+-----+----------+-----------+-----------------+------------+
| id|        name|state|      date|price_spent|is_correct_record|  name_upper|
+---+------------+-----+----------+-----------+-----------------+------------+
|  1| Alice Smith|   CA|2023-01-15|      125.5|             true| ALICE SMITH|
| 12|   Liam Blue|   CO|2023-12-05|       NULL|            false|   LIAM BLUE|
| 16|Peter Parker|   CA|2024-03-20|       NULL|            false|PETER PARKER|
+---+------------+-----+----------+-----------+-----------------+------------+



In [11]:
#handling float / integer values

df = df.withColumn('price_spent',col('price_spent').cast('float'))
df.printSchema()

df_filled = df.fillna({'price_spent':0})
df_filled.show()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- date: string (nullable = true)
 |-- price_spent: float (nullable = true)
 |-- is_correct_record: string (nullable = true)
 |-- name_upper: string (nullable = true)

+---+-------------+-----+----------+-----------+-----------------+-------------+
| id|         name|state|      date|price_spent|is_correct_record|   name_upper|
+---+-------------+-----+----------+-----------+-----------------+-------------+
|  1|  Alice Smith|   CA|2023-01-15|      125.5|             true|  ALICE SMITH|
|  2|  Bob Johnson|   NY|2023-02-20|       75.0|             true|  BOB JOHNSON|
|  3|Charlie Brown|   TX|2023-03-01|     200.25|             true|CHARLIE BROWN|
|  4| Diana Prince|   FL|2023-04-10|       50.0|             true| DIANA PRINCE|
|  5|    Eve Adams|   WA|2023-05-05|     300.75|             true|    EVE ADAMS|
|  6|  Frank White|   IL|2023-06-12|      99.99|             true|  FRAN

#Handle Date Column

In [12]:
df_handle_dates = df_filled.withColumn(
    'date' ,
    coalesce(
        try_to_timestamp(df_filled.date, lit('yyyy-MM-dd')),
        try_to_timestamp(df_filled.date, lit('yyyy/MM/dd')),
        try_to_timestamp(df_filled.date, lit('dd-mm-yyyy'))
    )
)

df_dates = df_handle_dates.withColumn( 'date', to_date(df_handle_dates.date))

In [13]:
df_dates.show()

+---+-------------+-----+----------+-----------+-----------------+-------------+
| id|         name|state|      date|price_spent|is_correct_record|   name_upper|
+---+-------------+-----+----------+-----------+-----------------+-------------+
|  1|  Alice Smith|   CA|2023-01-15|      125.5|             true|  ALICE SMITH|
|  2|  Bob Johnson|   NY|2023-02-20|       75.0|             true|  BOB JOHNSON|
|  3|Charlie Brown|   TX|2023-03-01|     200.25|             true|CHARLIE BROWN|
|  4| Diana Prince|   FL|2023-04-10|       50.0|             true| DIANA PRINCE|
|  5|    Eve Adams|   WA|2023-05-05|     300.75|             true|    EVE ADAMS|
|  6|  Frank White|   IL|2023-06-12|      99.99|             true|  FRANK WHITE|
|  7|    Grace Lee|   GA|2023-07-22|       15.2|             true|    GRACE LEE|
|  8|   Henry King|   AZ|2023-08-01|      180.0|             true|   HENRY KING|
|  9|    Ivy Queen|   PA|2023-09-01|       60.5|             true|    IVY QUEEN|
| 10|   Jack Black|   OH|202

In [14]:
spark.sql('show tables').show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [15]:
df_dates.createOrReplaceTempView('temp_customers')

In [16]:
spark.sql('select * from temp_customers').show()

+---+-------------+-----+----------+-----------+-----------------+-------------+
| id|         name|state|      date|price_spent|is_correct_record|   name_upper|
+---+-------------+-----+----------+-----------+-----------------+-------------+
|  1|  Alice Smith|   CA|2023-01-15|      125.5|             true|  ALICE SMITH|
|  2|  Bob Johnson|   NY|2023-02-20|       75.0|             true|  BOB JOHNSON|
|  3|Charlie Brown|   TX|2023-03-01|     200.25|             true|CHARLIE BROWN|
|  4| Diana Prince|   FL|2023-04-10|       50.0|             true| DIANA PRINCE|
|  5|    Eve Adams|   WA|2023-05-05|     300.75|             true|    EVE ADAMS|
|  6|  Frank White|   IL|2023-06-12|      99.99|             true|  FRANK WHITE|
|  7|    Grace Lee|   GA|2023-07-22|       15.2|             true|    GRACE LEE|
|  8|   Henry King|   AZ|2023-08-01|      180.0|             true|   HENRY KING|
|  9|    Ivy Queen|   PA|2023-09-01|       60.5|             true|    IVY QUEEN|
| 10|   Jack Black|   OH|202

#Global Temporary table


In [17]:
df.createOrReplaceGlobalTempView('temp_customer_raw')

In [18]:
spark.sql('show tables').show() # will only be able to view temporary tables

+---------+--------------+-----------+
|namespace|     tableName|isTemporary|
+---------+--------------+-----------+
|         |temp_customers|       true|
+---------+--------------+-----------+



In [19]:
spark.sql('show tables in global_temp').show()

+-----------+-----------------+-----------+
|  namespace|        tableName|isTemporary|
+-----------+-----------------+-----------+
|global_temp|temp_customer_raw|       true|
|           |   temp_customers|       true|
+-----------+-----------------+-----------+



In [20]:
spark_new = spark.newSession() # creating a new session in spark
spark_new.sql('show tables in global_temp').show() # the global tables will be accessible throughout various sessions

+-----------+-----------------+-----------+
|  namespace|        tableName|isTemporary|
+-----------+-----------------+-----------+
|global_temp|temp_customer_raw|       true|
+-----------+-----------------+-----------+



In [21]:
spark_new.sql('select * from global_temp.temp_customer_raw').show()

+---+-------------+-----+----------+-----------+-----------------+-------------+
| id|         name|state|      date|price_spent|is_correct_record|   name_upper|
+---+-------------+-----+----------+-----------+-----------------+-------------+
|  1|  Alice Smith|   CA|2023-01-15|      125.5|             true|  ALICE SMITH|
|  2|  Bob Johnson|   NY|2023-02-20|       75.0|             true|  BOB JOHNSON|
|  3|Charlie Brown|   TX|2023-03-01|     200.25|             true|CHARLIE BROWN|
|  4| Diana Prince|   FL|2023-04-10|       50.0|             true| DIANA PRINCE|
|  5|    Eve Adams|   WA|2023-05-05|     300.75|             true|    EVE ADAMS|
|  6|  Frank White|   IL|2023-06-12|      99.99|             true|  FRANK WHITE|
|  7|    Grace Lee|   GA|2023-07-22|       15.2|             true|    GRACE LEE|
|  8|   Henry King|   AZ|2023-08-01|      180.0|             true|   HENRY KING|
|  9|    Ivy Queen|   PA|2023/09/01|       60.5|             true|    IVY QUEEN|
| 10|   Jack Black|   OH|10-

#Persistent Table

In [22]:
df_dates.write.mode('overwrite').saveAsTable('customers_persistent')

In [23]:
spark_new.sql('show tables').show() #the persistent tables are shown in both the sessions just like global temp

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|  default|customers_persistent|      false|
+---------+--------------------+-----------+



In [24]:
spark.sql('describe extended customers_persistent').show(truncate=False)

+----------------------------+--------------------------------------------------+-------+
|col_name                    |data_type                                         |comment|
+----------------------------+--------------------------------------------------+-------+
|id                          |bigint                                            |NULL   |
|name                        |string                                            |NULL   |
|state                       |string                                            |NULL   |
|date                        |date                                              |NULL   |
|price_spent                 |float                                             |NULL   |
|is_correct_record           |string                                            |NULL   |
|name_upper                  |string                                            |NULL   |
|                            |                                                  |       |
|# Detaile